In [ ]:
import boto3

Setting up the s3 client and retrieving available buckets

In [ ]:
s3 = boto3.client('s3')

In [ ]:
s3.list_buckets()

# Create a new Bucket

Provide a bucket name and a region to create a new bucket

In [ ]:
bucket_name = 'soosisbucket'

In [ ]:
def create_bucket(bucket_name):i
    """Create an S3 bucket in a specified region"""
    try:
        s3.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={
            'LocationConstraint': "eu-north-1"
        })
        print(f"Bucket {bucket_name} created successfully.")
    except Exception as e:
        print(f"Error creating bucket: {e}")

In [ ]:
create_bucket(bucket_name)

# Upload single file to S3

Selecting a file and uploading it

In [ ]:
import os

def upload_file_to_s3(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket"""
    if object_name is None:
        object_name = os.path.basename(file_name)
    
    try:
        s3.upload_file(file_name, bucket, object_name)
        print(f"File {file_name} uploaded to {bucket}/{object_name}.")
    except Exception as e:
        print(f"Error uploading file: {e}")

In [ ]:
upload_file_to_s3('boto_s3.ipynb', bucket_name)

In [ ]:
response = s3.list_objects_v2(Bucket=bucket_name)
for obj in response["Contents"]:
    print(obj["Key"])

# Download S3 File to Local System

Downloading a single file from the S3 bucket

In [ ]:
def download_file_from_s3(bucket_name, object_name, file_name):
    """Download a file from an S3 bucket"""
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    try:
        s3.download_file(bucket_name, object_name, file_name)
        print(f"File {object_name} downloaded from {bucket_name} to {file_name}.")
    except Exception as e:
        print(f"Error downloading file: {e}")

In [ ]:
download_file_from_s3(bucket_name, 'boto_s3.ipynb', 'data_download/downloaded_boto_s3.ipynb')

# Upload Folder to S3

Uploading a whole folder to S3

In [ ]:
def upload_directory_to_s3(directory, bucket_name, s3_prefix):
    """Upload a directory to an S3 bucket"""
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            relative_path = os.path.relpath(file_path, directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(file_path, relative_path, s3_key)
            upload_file_to_s3(file_path, bucket_name, s3_key)

In [ ]:
upload_directory_to_s3('data', bucket_name, "s3_data")

# Download S3 Folder to Local System

Downloading a folder from S3 by iterating over every file available

In [ ]:
local_path = 'data_download'
s3_prefix = 's3_data/'


def download_dir_from_s3(bucket_name, s3_prefix):
    """Download a directory from an S3 bucket"""
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    paginator = s3.get_paginator("list_objects_v2")
    for result in paginator.paginate(Bucket=bucket_name, Prefix=s3_prefix):
        if "Contents" in result:
            for obj in result["Contents"]:
                s3_key = obj["Key"]
                print(s3_key)
                local_file_path = os.path.join(local_path, s3_key.replace(s3_prefix, ""))
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

                s3.download_file(bucket_name, s3_key, local_file_path)
                print(f"Downloaded {obj['Key']} to {local_file_path}.")

In [ ]:
download_dir_from_s3(bucket_name, s3_prefix)

# Delete all Files in an S3 Bucket

Delete all files to be able to delete the S3 bucket afterwards

In [ ]:
def delete_object_in_s3():
    """Delete all files in an S3 bucket"""
    paginator = s3.get_paginator("list_objects_v2")
    for result in paginator.paginate(Bucket=bucket_name):
        # print("Result: ", result)
        if "Contents" in result:
            for obj in result["Contents"]:
                s3.delete_object(Bucket=bucket_name, Key=obj["Key"])
                print(f"Deleted {obj['Key']} from {bucket_name}.")

In [ ]:
delete_object_in_s3()

In [ ]:
s3.delete_bucket(Bucket=bucket_name)
print(f"Bucket {bucket_name} deleted successfully.")